# Mejora del enrutamiento diferido (la pereza del enrutador)

<a href="https://colab.research.google.com/github/JunSuzukiJapan/SynapticRouter/blob/main/notebooks/16_lazy_routing_prevention_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Descripción general
En este cuaderno, comparamos y evaluamos dos enfoques para mitigar el problema "solo se utilizan unas pocas sinapsis (enrutamiento diferido/colapso de representación)" que tiende a ocurrir en SRA (arquitectura de enrutamiento sináptico).

1. **Límite de capacidad (restricción estricta)**: coloque un límite superior en la cantidad de tokens procesados ​​por sinapsis y elimine (pase) a la fuerza cualquier desbordamiento.
2. **Control de entropía (restricción suave)**: manipule la función de pérdida para fomentar que "el uso general sea uniforme (maximizar la entropía)" mientras que "cada punto de datos elige una sinapsis sin dudarlo (minimizar la entropía)".

Entrenamos modelos con cada método solo o combinado, y medimos la **Entropía de enrutamiento (uniformidad de la distribución de uso)** y la **Proporción de sinapsis muertas (la proporción de sinapsis muertas)**.

In [ ]:
# Setup for running on Google Colab
import os
import sys

if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    sys.path.append(os.getcwd())
    sys.path.append(os.path.join(os.getcwd(), 'src'))
    print("Setup completed on Google Colab.")
else:
    sys.path.append(os.path.abspath('..'))
    sys.path.append(os.path.join(os.path.abspath('..'), 'src'))
    print("Running locally.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import matplotlib.pyplot as plt

from src.sra_gpu_models import MoESRAModel, MoESRABlock
from src.sra_experiment import make_optimizer, load_balance_loss
from src.constants import PAD, BOS, EOS

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

# ===== Task definitions (excerpt) =====
VOCAB_SIZE = 128
def encode(text): return [BOS] + [ord(c) for c in text] + [EOS]
def pad_to(seq, length): return seq[:length] + [PAD] * max(0, length - len(seq))

WORDS = ["hello", "world", "python", "learn", "great", "smart", "open", "code"]
VARS = ["x", "y", "z", "n", "val", "res"]
BASES = ['A', 'T', 'G', 'C']
COMP  = {'A':'T', 'T':'A', 'G':'C', 'C':'G'}

def nl_upper(): w = random.choice(WORDS); return w, w.upper()
def code_indent(): v = random.choice(VARS); n = random.randint(1,9); return f"return {v} + {n}", f"    return {v} + {n}"
def math_add(): a, b = random.randint(1,19), random.randint(1,19); return f"{a}+{b}=", str(a+b)
def dna_complement(): s = ''.join(random.choices(BASES, k=5)); return s, ''.join(COMP[c] for c in s)

ALL_TASKS = {
    "nl_upper": nl_upper, "code_indent": code_indent,
    "math_add": math_add, "dna_complement": dna_complement
}

MAX_SEQ_LEN = 24
DIM = 64
LAYERS = 2
K = 2
SYN_HIDDEN = 128

def make_multitask_batch(tasks, batch_size):
    xs, ys = [], []
    for _ in range(batch_size):
        task_name = random.choice(list(tasks.keys()))
        inp_str, out_str = tasks[task_name]()
        xs.append(pad_to(encode(inp_str), MAX_SEQ_LEN))
        ys.append(pad_to(encode(out_str), MAX_SEQ_LEN))
    return torch.tensor(xs, dtype=torch.long, device=device), torch.tensor(ys, dtype=torch.long, device=device)

In [ ]:
def calculate_routing_metrics(model, num_synapses, samples=100):
    model.eval()
    all_usage_counts = torch.zeros(num_synapses, device=device)
    total_tokens = 0
    all_probs = []
    
    with torch.no_grad():
        for _ in range(samples):
            x, y = make_multitask_batch(ALL_TASKS, 128)
            y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
            valid_mask = (torch.cat([x, y_in], dim=1) != PAD)
            
            _, router_logits, _ = model(x, y_in)
            logits = router_logits[-1] # (B, T, num_synapses)
            probs = F.softmax(logits, dim=-1)
            
            # Entropy
            valid_probs = probs[valid_mask]
            all_probs.append(valid_probs)
            
            # Dead Synapse Ratio
            _, topk = torch.topk(logits, K, dim=-1)
            valid_topk = topk[valid_mask]
            for k_idx in range(K):
                all_usage_counts.index_add_(0, valid_topk[:, k_idx], torch.ones(valid_topk.size(0), device=device))
            total_tokens += valid_topk.size(0)
            
    # Routing Entropy
    cat_probs = torch.cat(all_probs, dim=0) # (Total_Tokens, num_synapses)
    mean_probs = cat_probs.mean(dim=0)
    routing_entropy = -(mean_probs * torch.log(mean_probs + 1e-9)).sum().item()
    
    # Dead Synapse Ratio (< 1% of expected uniform usage)
    expected_usage = (total_tokens * K) / num_synapses
    dead_threshold = expected_usage * 0.01
    dead_synapses = (all_usage_counts < dead_threshold).sum().item()
    dead_ratio = (dead_synapses / num_synapses) * 100
    
    return routing_entropy, dead_ratio

In [ ]:
class MoESRABlockWithCapacity(MoESRABlock):
    def forward(self, h, dense=False, key_padding_mask=None, encoder_len=0, capacity_factor=None):
        base = h
        B, T, D = h.shape
        
        # 1. Shared Attention
        attn_mask = torch.zeros((T, T), dtype=torch.bool, device=h.device)
        if encoder_len > 0:
            attn_mask[:encoder_len, encoder_len:] = True
            future_target = torch.triu(torch.ones((T - encoder_len, T - encoder_len), dtype=torch.bool, device=h.device), diagonal=1)
            attn_mask[encoder_len:, encoder_len:] = future_target
            
        attn_out, _ = self.attn(h, h, h, attn_mask=attn_mask, key_padding_mask=key_padding_mask, need_weights=False)
        h = self.norm1(h + attn_out)
        
        # 2. MoE Routing
        h_routed = h
        h_routed = self.norm2(h_routed)
        k_override = self.num_synapses if dense else self.k
        idx, weights, logits = self.router(h_routed, k_override=k_override)
        
        # Gather/Scatter
        h_flat = h_routed.view(B*T, D)
        idx_flat = idx.view(B*T, -1)
        weights_flat = weights.view(B*T, -1)
        out_flat = torch.zeros_like(h_flat)
        
        if capacity_factor is not None:
            capacity_limit = int((B * T * capacity_factor) / self.num_synapses)
        else:
            capacity_limit = B * T
            
        for e in range(self.num_synapses):
            mask = (idx_flat == e) # (B*T, k)
            if not mask.any(): continue
            
            token_indices = mask.any(dim=-1).nonzero().squeeze(-1)
            
            # --- Capacity Limit Drop ---
            if len(token_indices) > capacity_limit:
                token_indices = token_indices[:capacity_limit]
            # ---------------------------
            
            h_sub = h_flat[token_indices]
            
            w1_ex = self.w1[e]
            b1_ex = self.b1[e]
            w2_ex = self.w2[e]
            b2_ex = self.b2[e]
            state_ex = self.state[e]
            
            hidden = F.gelu(torch.matmul(h_sub, w1_ex) + b1_ex)
            expert_out = torch.matmul(hidden, w2_ex) + b2_ex + state_ex
            
            kept_mask = torch.zeros_like(mask)
            kept_mask[token_indices] = mask[token_indices]
            expert_weights = weights_flat[kept_mask]
            
            out_flat[token_indices] += expert_out * expert_weights.unsqueeze(-1)
            
        out = out_flat.view(B, T, D)
        dummy_syn_outs = [torch.zeros(B, T, D, device=h.device) for _ in range(self.num_synapses)]
        return base + out, logits, dummy_syn_outs

class MoESRAModelWithCapacity(MoESRAModel):
    def __init__(self, vocab_size, dim, layers, num_synapses, k, syn_hidden):
        super().__init__(vocab_size, dim, layers, num_synapses, k, syn_hidden)
        self.blocks = nn.ModuleList([MoESRABlockWithCapacity(dim, num_synapses, k, syn_hidden) for _ in range(layers)])
        
    def forward(self, x, y_in, dense=False, capacity_factor=None):
        seq = torch.cat([x, y_in], dim=1)
        mask = seq == PAD
        segment_ids = torch.cat([torch.zeros_like(x), torch.ones_like(y_in)], dim=1)
        target_rel_pos = torch.cat([torch.zeros_like(x), torch.arange(1, y_in.size(1) + 1, device=seq.device).unsqueeze(0).repeat(x.size(0), 1)], dim=1)
        h = self.embed(seq) + self.pos[:, :seq.size(1)] + self.seg(segment_ids) + self.rel_pos(target_rel_pos)
        
        router_logits = []
        all_synapse_outputs = []
        for block in self.blocks:
            h, logits, syn_outs = block(h, dense=dense, key_padding_mask=mask, encoder_len=x.size(1), capacity_factor=capacity_factor)
            router_logits.append(logits)
            all_synapse_outputs.append(syn_outs)
        return self.out(h[:, x.size(1):]), router_logits, all_synapse_outputs

In [ ]:
def calculate_entropy_loss(router_logits):
    loss = 0.0
    for logits in router_logits:
        probs = F.softmax(logits, dim=-1)
        mean_probs = probs.mean(dim=(0, 1))
        
        mean_entropy = -(mean_probs * torch.log(mean_probs + 1e-9)).sum()
        individual_entropy = -(probs * torch.log(probs + 1e-9)).sum(dim=-1).mean()
        
        # Minimize individual_entropy (be confident), Maximize mean_entropy (be uniform)
        loss += (0.1 * individual_entropy) - (0.1 * mean_entropy)
    return loss

def train_model_experiment(num_synapses, epochs=1000, capacity_factor=None, use_entropy_loss=False):
    model = MoESRAModelWithCapacity(vocab_size=128, dim=DIM, layers=LAYERS, num_synapses=num_synapses, k=K, syn_hidden=SYN_HIDDEN).to(device)
    optimizer = make_optimizer(model, lr=1e-3)
    model.train()
    
    for epoch in range(epochs):
        x, y = make_multitask_batch(ALL_TASKS, 128)
        optimizer.zero_grad()
        y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        
        outputs, routing_weights, _ = model(x, y_in, capacity_factor=capacity_factor)
        
        ce_loss = F.cross_entropy(outputs.reshape(-1, 128), y.reshape(-1), ignore_index=PAD)
        lb_loss = load_balance_loss(routing_weights) * 0.1
        total_loss = ce_loss + lb_loss
        
        if use_entropy_loss:
            ent_loss = calculate_entropy_loss(routing_weights)
            total_loss += ent_loss
            
        total_loss.backward()
        # Prevent exploding gradients due to negative loss components
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if (epoch + 1) % 500 == 0:
            print(f"Epoch {epoch+1}/{epochs} | CE Loss: {ce_loss.item():.4f}")
            
    return model

In [ ]:
num_synapses = 128
epochs = 1000

conditions = [
    {"name": "1. Baseline", "capacity_factor": None, "use_entropy_loss": False},
    {"name": "2. +Capacity Limit", "capacity_factor": 1.5, "use_entropy_loss": False},
    {"name": "3. +Entropy Control", "capacity_factor": None, "use_entropy_loss": True},
    {"name": "4. +Both", "capacity_factor": 1.5, "use_entropy_loss": True}
]

results = {}

for cond in conditions:
    print(f"\n--- Training Condition: {cond['name']} ---")
    model = train_model_experiment(
        num_synapses=num_synapses, 
        epochs=epochs, 
        capacity_factor=cond["capacity_factor"], 
        use_entropy_loss=cond["use_entropy_loss"]
    )
    
    print(f"Calculating metrics for {cond['name']}...")
    entropy, dead_ratio = calculate_routing_metrics(model, num_synapses)
    print(f"Routing Entropy: {entropy:.3f} | Dead Synapse Ratio: {dead_ratio:.1f}%")
    
    results[cond["name"]] = {
        "entropy": entropy,
        "dead_ratio": dead_ratio
    }

In [ ]:
# Visualization of the results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
names = list(results.keys())
entropies = [results[n]["entropy"] for n in names]
dead_ratios = [results[n]["dead_ratio"] for n in names]

# Max possible entropy
max_entropy = float(np.log(num_synapses))

colors = ['gray', 'orange', 'green', 'blue']

bars1 = ax1.bar(names, entropies, color=colors)
ax1.axhline(max_entropy, color='red', linestyle='--', label=f'Max Entropy ({max_entropy:.2f})')
ax1.set_title("Routing Entropy (Higher is better/more uniform)")
ax1.set_ylabel("Entropy")
ax1.tick_params(axis='x', rotation=15)
ax1.legend()

bars2 = ax2.bar(names, dead_ratios, color=colors)
ax2.set_title("Dead Synapse Ratio (Lower is better)")
ax2.set_ylabel("Ratio (%)")
ax2.set_ylim(0, 105)
ax2.tick_params(axis='x', rotation=15)

# Display values above the bars
for bar in bars1:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + 0.1, f'{yval:.2f}', ha='center', va='bottom')
    
for bar in bars2:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, yval + 2, f'{yval:.1f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Discusión
A partir de los resultados experimentales, podemos comparar la efectividad de cada enfoque.

- **Línea de base**: Sin ninguna contramedida, incluso con una gran capacidad de 128 sinapsis, el procesamiento se concentra en unas pocas sinapsis y muchas de ellas terminan muertas (alta proporción de sinapsis muertas).
- **Límite de capacidad**: cuando establecemos un límite superior de "estilo de hardware" para "cuánto puede procesar una sinapsis", el enrutador se ve obligado a explorar otras sinapsis libres, por lo que las sinapsis muertas se reducen drásticamente y la entropía mejora.
- **Control de entropía**: la guía a través de una restricción suave (función de pérdida) también lleva al modelo a aprender a usar las sinapsis de manera uniforme en general, lo que produce una mejora significativa.
- **Ambos**: al combinar los dos, logramos la distribución de enrutamiento uniforme y más utilizada en todas las sinapsis (un estado cercano a la entropía máxima).